In [1]:
import pandas as pd

In [2]:
df_media = pd.read_csv('C:/Users/Josue/4GA.Datascience/data/raw/Grupo1.csv')

In [3]:
#Eliminamos columnas del df que no interesen
columnas_a_eliminar = ['name', 'essround', 'edition', 'proddate', 'idno', 'dweight', 'pspwght', 'pweight', 'anweight', 'prob', 'stratum', 'psu']
df_media.drop(columnas_a_eliminar, axis=1, inplace=True)

In [4]:
#Obtenemos una lista con los nombres de las columnas restantes.
nombres_columnas = df_media.columns.tolist()
print(nombres_columnas)

['cntry', 'pplfair', 'pplhlp', 'ppltrst', 'lawobey', 'lrscale', 'polintr', 'stfdem', 'stfeco', 'stfgov', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt', 'trstsci', 'imbgeco', 'imwbcnt', 'happy', 'rlgdgr']


In [5]:
#Obtenemos una lista con los paises de las muestras de las filas.
valores_unicos_cntry = df_media['cntry'].unique()
print(valores_unicos_cntry)

['ES' 'IT' 'PT' 'HR']


In [6]:
#Guardamos en un dataframe nuestra tabla para asignar grupo y valores clave de la columna cntry.
import sqlite3
import pandas as pd


db_path = 'C:/Users/Josue/4GA.DataScience/src/EcoUE.db'
try:
    conn = sqlite3.connect(db_path)
    query = "SELECT * FROM ecoeu"
    df_ecoeu = pd.read_sql_query(query, conn)
    print(df_ecoeu)
except sqlite3.Error as e:
    print(f"Error al conectar o consultar la base de datos: {e}")

finally:
    
    if conn:
        conn.close()

           cntry       PIB  Inflation    sma       cntrycat  \
0       Alemania   54343.2        5.9  60867           Rico   
1        Austria   56033.6        7.8  57082           Rico   
2        Bélgica   54700.9        4.0  59285           Rico   
3         Chipre   36551.4        3.5  26689  Media Europea   
4        Croacia   21865.5        7.9  17714  Media Europea   
5      Eslovenia   32610.1        7.4  26667  Media Europea   
6         España   33509.0        3.5  30237  Media Europea   
7        Estonia   30133.3        9.2  21595  Media Europea   
8      Finlandia   52925.7        6.3  53310           Rico   
9        Francia   44690.9        4.9  43438  Media Europea   
10        Grecia   23400.7        3.5  23536  Media Europea   
11       Irlanda  103887.8        6.3  59899           Rico   
12        Italia   39003.3        5.6  33492  Media Europea   
13       Letonia   22502.8        8.9  18559  Media Europea   
14      Lituania   27786.0        9.1  23409  Media Eur

In [7]:
cntrymap= {
    'ES':'España',
    'IT':'Italia',
    'PT':'Portugal',
    'HR':'Croacia',
}


In [8]:
df_media['cntry'] = df_media['cntry'].map(cntrymap)

In [9]:
df_media = pd.merge(df_media, df_ecoeu[['cntry', 'cntrycat_factorizado']], on='cntry', how='left')

In [10]:
print(df_media[['cntry', 'cntrycat_factorizado']])

          cntry  cntrycat_factorizado
0        España                     1
1        España                     1
2        España                     1
3        España                     1
4        España                     1
...         ...                   ...
61686  Portugal                     1
61687  Portugal                     1
61688  Portugal                     1
61689  Portugal                     1
61690  Portugal                     1

[61691 rows x 2 columns]


In [11]:
df_media.info()
import numpy as np, random
df_media['lawobey'] = np.nan

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61691 entries, 0 to 61690
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cntry                 61691 non-null  object 
 1   pplfair               61691 non-null  int64  
 2   pplhlp                61691 non-null  int64  
 3   ppltrst               61691 non-null  int64  
 4   lawobey               4447 non-null   float64
 5   lrscale               61691 non-null  int64  
 6   polintr               61691 non-null  int64  
 7   stfdem                61691 non-null  int64  
 8   stfeco                61691 non-null  int64  
 9   stfgov                61691 non-null  int64  
 10  trstlgl               61691 non-null  int64  
 11  trstplc               61691 non-null  int64  
 12  trstplt               61691 non-null  int64  
 13  trstprl               61691 non-null  int64  
 14  trstprt               57244 non-null  float64
 15  trstsci            

In [12]:
import pandas as pd
import numpy as np

def marcar_atipicos_ordinales_nan_contador(df):
    """
    Marca los valores atípicos en columnas ordinales con NaN y cuenta los atípicos en 'contadornegativo'.


    """

    df_modificado = df.copy()
    columnas_numericas = df_modificado.select_dtypes(include=np.number).columns
    df_modificado['contadornegativo'] = 0  # Inicializa la columna 'contadornegativo'

    for columna in columnas_numericas:
        max_valor = df_modificado[columna].max()

        if max_valor == 9:
            # Encuentra los índices de los valores atípicos
            indices_atipicos = df_modificado.loc[df_modificado[columna].isin([6, 7, 8, 9])].index

            # Marca los atípicos con NaN
            df_modificado.loc[indices_atipicos, columna] = np.nan

            # Suma +1 a 'contadornegativo' para las filas con atípicos
            df_modificado.loc[indices_atipicos, 'contadornegativo'] += 1

        elif max_valor == 5:
            # Encuentra los índices de los valores atípicos
            indices_atipicos = df_modificado.loc[df_modificado[columna].isin([0, 5])].index

            # Marca los atípicos con NaN
            df_modificado.loc[indices_atipicos, columna] = np.nan

            # Suma +1 a 'contadornegativo' para las filas con atípicos
            df_modificado.loc[indices_atipicos, 'contadornegativo'] += 1

    return df_modificado

# Ejemplo de uso:
# Supongamos que tienes tu DataFrame llamado df
df_media = marcar_atipicos_ordinales_nan_contador(df_media)

# Imprime la forma del DataFrame original y del DataFrame modificado
print(f"Forma del DataFrame original: {df_media.shape}")
print(f"Forma del DataFrame modificado: {df_media.shape}")

# Imprime la cantidad de valores NaN por columna en el DataFrame modificado
print(df_media.isnull().sum())

# Imprime la columna 'contadornegativo'
print(df_media['contadornegativo'].head(10)) #imprime los primeros 10 valores

Forma del DataFrame original: (61691, 22)
Forma del DataFrame modificado: (61691, 22)
cntry                       0
pplfair                     0
pplhlp                      0
ppltrst                     0
lawobey                 61691
lrscale                     0
polintr                   185
stfdem                      0
stfeco                      0
stfgov                      0
trstlgl                     0
trstplc                     0
trstplt                     0
trstprl                     0
trstprt                  4447
trstsci                 53338
imbgeco                     0
imwbcnt                     0
happy                       0
rlgdgr                      0
cntrycat_factorizado        0
contadornegativo            0
dtype: int64
0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
Name: contadornegativo, dtype: int64


In [13]:
import pandas as pd
import numpy as np

def marcar_atipicos_ordinales_nan_contador_10_acumulativo_df2(df):
    """
    Marca los valores atípicos 66, 77, 88, 99 en columnas ordinales con NaN y acumula los atípicos en 'contadornegativo' .

    """

    df_modificado2 = df.copy()
    columnas_numericas = df_modificado2.select_dtypes(include=np.number).columns

    # Asegurarse de que 'contadornegativo' existe, si no, inicializarla.
    if 'contadornegativo' not in df_modificado2.columns:
        df_modificado2['contadornegativo'] = 0

    for columna in columnas_numericas:
        max_valor = df_modificado2[columna].max()

        if max_valor == 99:
            # Encuentra los índices de los valores atípicos
            indices_atipicos = df_modificado2.loc[df_modificado2[columna].isin([66, 77, 88, 99])].index

            # Marca los atípicos con NaN
            df_modificado2.loc[indices_atipicos, columna] = np.nan

            # Suma +1 a 'contadornegativo' para las filas con atípicos
            df_modificado2.loc[indices_atipicos, 'contadornegativo'] += 1

        elif max_valor == 10:
            # Encuentra los índices de los valores atípicos
            indices_atipicos = df_modificado2.loc[df_modificado2[columna].isin([0, 10])].index

            # Marca los atípicos con NaN
            df_modificado2.loc[indices_atipicos, columna] = np.nan

            # Suma +1 a 'contadornegativo' para las filas con atípicos
            df_modificado2.loc[indices_atipicos, 'contadornegativo'] += 1

    return df_modificado2

# Ejemplo de uso:
# Supongamos que tienes tu DataFrame llamado df_modificado2
df_media = marcar_atipicos_ordinales_nan_contador_10_acumulativo_df2(df_media)

# Imprime la forma del DataFrame original y del DataFrame modificado
print(f"Forma del DataFrame original: {df_media.shape}")


# Imprime la cantidad de valores NaN por columna en el DataFrame modificado
print(df_media.isnull().sum())

# Imprime la columna 'contadornegativo'
print(df_media['contadornegativo'].head(10))

Forma del DataFrame original: (61691, 22)
cntry                       0
pplfair                   584
pplhlp                    460
ppltrst                   279
lawobey                 61691
lrscale                 12227
polintr                   185
stfdem                   2610
stfeco                   1416
stfgov                   2240
trstlgl                  1847
trstplc                   780
trstplt                  1149
trstprl                  2409
trstprt                  5618
trstsci                 53521
imbgeco                  3335
imwbcnt                  2981
happy                     293
rlgdgr                    654
cntrycat_factorizado        0
contadornegativo            0
dtype: int64
0    0
1    0
2    0
3    0
4    3
5    1
6    2
7    0
8    0
9    0
Name: contadornegativo, dtype: int64


In [23]:
import pandas as pd
import numpy as np

dataspa = {
    'lawobey': {
        1: 0.0222,
        2: 0.435,
        3: 0.23,
        4: 0.094,
        5: 0.02,
    },
    'trstsci': {
        0: 0.011,
        1: 0.003,
        2: 0.018,
        3: 0.028,
        4: 0.039,
        5: 0.075,
        6: 0.132,
        7: 0.217,
        8: 0.252,
        9: 0.136,
        10: 0.088,
    }
}

def rngpond(df, dataspa, cntry_value, missing_threshold):

    df_cntry = df[df['cntry'] == cntry_value].copy()  

    for columna, pesos in dataspa.items():
        if columna in df_cntry.columns and df_cntry[columna].isnull().any():
            valores = np.array(list(pesos.keys()))
            pesos_ponderados = np.array(list(pesos.values()))

            
            suma_pesos = np.sum(pesos_ponderados)
            pesos_normalizados = pesos_ponderados / suma_pesos

            for index, row in df_cntry[df_cntry[columna].isnull()].iterrows():
                
                valor_aleatorio = np.random.choice(valores, p=pesos_normalizados)

                
                df_cntry.loc[index, columna] = valor_aleatorio

    
    for columna in df_cntry.columns:
        if df_cntry[columna].isnull().any():
            missing_percentage = df_cntry[columna].isnull().sum() / len(df_cntry)
            if missing_percentage < missing_threshold:
                mode_value = df_cntry[columna].mode()[0]
                df_cntry[columna].fillna(mode_value, inplace=True)

    
    df.loc[df['cntry'] == cntry_value] = df_cntry 


rngpond(df_media, dataspa, 'España', missing_threshold=0.3)  # Ejemplo para España con umbral del 20%
print(df_media[df_media['cntry'] == 'España'].isnull().sum())

cntry                   0
pplfair                 0
pplhlp                  0
ppltrst                 0
lawobey                 0
lrscale                 0
polintr                 0
stfdem                  0
stfeco                  0
stfgov                  0
trstlgl                 0
trstplc                 0
trstplt                 0
trstprl                 0
trstprt                 0
trstsci                 0
imbgeco                 0
imwbcnt                 0
happy                   0
rlgdgr                  0
cntrycat_factorizado    0
contadornegativo        0
dtype: int64


In [24]:
df_media.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61691 entries, 0 to 61690
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cntry                 61691 non-null  object 
 1   pplfair               61691 non-null  float64
 2   pplhlp                61691 non-null  float64
 3   ppltrst               61691 non-null  float64
 4   lawobey               61691 non-null  float64
 5   lrscale               53501 non-null  float64
 6   polintr               61691 non-null  float64
 7   stfdem                61691 non-null  float64
 8   stfeco                61691 non-null  float64
 9   stfgov                61691 non-null  float64
 10  trstlgl               61691 non-null  float64
 11  trstplc               61691 non-null  float64
 12  trstplt               61691 non-null  float64
 13  trstprl               61691 non-null  float64
 14  trstprt               61691 non-null  float64
 15  trstsci            

In [25]:
valores_unicos_cntry = df_media['cntry'].unique()
print(valores_unicos_cntry)

['España' 'Italia' 'Portugal' 'Croacia']


In [ ]:
import pandas as pd
import numpy as np

datacro = {
    'lawobey': {
        1: 0.487,
        2: 0.391,
        3: 0.082,
        4: 0.028,
        5: 0.011,
    },
    'trstsci': {
        0: 0.038,
        1: 0.017,
        2: 0.031,
        3: 0.05,
        4: 0.057,
        5: 0.144,
        6: 0.089,
        7: 0.158,
        8: 0.182,
        9: 0.116,
        10: 0.118,
    }
}
rngpond(df_media, datacro, 'Croacia', missing_threshold=0.3) 
print(df_media[df_media['cntry'] == 'Croacia'].isnull().sum())

cntry                   0
pplfair                 0
pplhlp                  0
ppltrst                 0
lawobey                 0
lrscale                 0
polintr                 0
stfdem                  0
stfeco                  0
stfgov                  0
trstlgl                 0
trstplc                 0
trstplt                 0
trstprl                 0
trstprt                 0
trstsci                 0
imbgeco                 0
imwbcnt                 0
happy                   0
rlgdgr                  0
cntrycat_factorizado    0
contadornegativo        0
dtype: int64


In [30]:
print(valores_unicos_cntry)
df_media.info()

['España' 'Italia' 'Portugal' 'Croacia']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61691 entries, 0 to 61690
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cntry                 61691 non-null  object 
 1   pplfair               61691 non-null  float64
 2   pplhlp                61691 non-null  float64
 3   ppltrst               61691 non-null  float64
 4   lawobey               61691 non-null  float64
 5   lrscale               53501 non-null  float64
 6   polintr               61691 non-null  float64
 7   stfdem                61691 non-null  float64
 8   stfeco                61691 non-null  float64
 9   stfgov                61691 non-null  float64
 10  trstlgl               61691 non-null  float64
 11  trstplc               61691 non-null  float64
 12  trstplt               61691 non-null  float64
 13  trstprl               61691 non-null  float64
 14  trstprt               61691 n

In [ ]:
import pandas as pd
import numpy as np

dataita= {
    'lawobey': {
        1: 0.508,
        2: 0.394,
        3: 0.073,
        4: 0.019,
        5: 0.006,
    },
    'trstsci': {
        0: 0.01,
        1: 0.003,
        2: 0.018,
        3: 0.029,
        4: 0.038,
        5: 0.073,
        6: 0.135,
        7: 0.217,
        8: 0.25,
        9: 0.134,
        10: 0.092,
    }
}
rngpond(df_media, dataita, 'Italia', missing_threshold=0.3) 
print(df_media[df_media['cntry'] == 'Italia'].isnull().sum())

cntry                   0
pplfair                 0
pplhlp                  0
ppltrst                 0
lawobey                 0
lrscale                 0
polintr                 0
stfdem                  0
stfeco                  0
stfgov                  0
trstlgl                 0
trstplc                 0
trstplt                 0
trstprl                 0
trstprt                 0
trstsci                 0
imbgeco                 0
imwbcnt                 0
happy                   0
rlgdgr                  0
cntrycat_factorizado    0
contadornegativo        0
dtype: int64


C:\Users\Josue\AppData\Local\Temp\ipykernel_11588\1385120080.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cntry[columna].fillna(mode_value, inplace=True)


In [32]:
print(valores_unicos_cntry)
df_media.info()

['España' 'Italia' 'Portugal' 'Croacia']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61691 entries, 0 to 61690
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cntry                 61691 non-null  object 
 1   pplfair               61691 non-null  float64
 2   pplhlp                61691 non-null  float64
 3   ppltrst               61691 non-null  float64
 4   lawobey               61691 non-null  float64
 5   lrscale               56989 non-null  float64
 6   polintr               61691 non-null  float64
 7   stfdem                61691 non-null  float64
 8   stfeco                61691 non-null  float64
 9   stfgov                61691 non-null  float64
 10  trstlgl               61691 non-null  float64
 11  trstplc               61691 non-null  float64
 12  trstplt               61691 non-null  float64
 13  trstprl               61691 non-null  float64
 14  trstprt               61691 n

In [ ]:
import pandas as pd
import numpy as np

datapor= {
    'lawobey': {
        1: 0.194,
        2: 0.489,
        3: 0.214,
        4: 0.094,
        5: 0.009,
    },
    'trstsci': {
        0: 0.011,
        1: 0.004,
        2: 0.007,
        3: 0.014,
        4: 0.025,
        5: 0.114,
        6: 0.082,
        7: 0.145,
        8: 0.285,
        9: 0.166,
        10: 0.147,
    }
}
rngpond(df_media, datapor, 'Portugal', missing_threshold=0.3) 
print(df_media[df_media['cntry'] == 'Portugal'].isnull().sum())

cntry                   0
pplfair                 0
pplhlp                  0
ppltrst                 0
lawobey                 0
lrscale                 0
polintr                 0
stfdem                  0
stfeco                  0
stfgov                  0
trstlgl                 0
trstplc                 0
trstplt                 0
trstprl                 0
trstprt                 0
trstsci                 0
imbgeco                 0
imwbcnt                 0
happy                   0
rlgdgr                  0
cntrycat_factorizado    0
contadornegativo        0
dtype: int64


C:\Users\Josue\AppData\Local\Temp\ipykernel_11588\1385120080.py:53: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cntry[columna].fillna(mode_value, inplace=True)


In [22]:
df_media.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61691 entries, 0 to 61690
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cntry                 61691 non-null  object 
 1   pplfair               61691 non-null  float64
 2   pplhlp                61691 non-null  float64
 3   ppltrst               61691 non-null  float64
 4   lawobey               61691 non-null  float64
 5   lrscale               53501 non-null  float64
 6   polintr               61691 non-null  float64
 7   stfdem                61691 non-null  float64
 8   stfeco                61691 non-null  float64
 9   stfgov                61691 non-null  float64
 10  trstlgl               61691 non-null  float64
 11  trstplc               61691 non-null  float64
 12  trstplt               61691 non-null  float64
 13  trstprl               61691 non-null  float64
 14  trstprt               61691 non-null  float64
 15  trstsci            

In [34]:
import pandas as pd
import numpy as np

def contar_neutros_ponderado(df):
    """
    Crea una columna 'ContadorNeutro' con neutralidad ponderada para diferentes escalas.

    Args:
        df (pd.DataFrame): El DataFrame.

    Returns:
        pd.DataFrame: El DataFrame con la nueva columna 'ContadorNeutro'.
    """

    df_modificado = df.copy()
    df_modificado['ContadorNeutro'] = 0.0

    for columna in df_modificado.select_dtypes(include=np.number).columns:
        valores_unicos = df_modificado[columna].dropna().unique()
        num_categorias = len(valores_unicos)

        if num_categorias > 2:
            if num_categorias == 3:
                valor_central = np.median(valores_unicos)
                df_modificado['ContadorNeutro'] += np.where(df_modificado[columna] == valor_central, 0.25, 0)
            elif num_categorias == 4:
                cuartiles = np.quantile(valores_unicos, [0.25, 0.75])
                df_modificado['ContadorNeutro'] += np.where((df_modificado[columna] >= cuartiles[0]) & (df_modificado[columna] <= cuartiles[1]), 0.25, 0)
            elif num_categorias == 5:
                valor_central = np.median(valores_unicos)
                df_modificado['ContadorNeutro'] += np.where(df_modificado[columna] == valor_central, 0.5, 0)
            elif num_categorias == 6:
                sextiles = np.quantile(valores_unicos, [1/3, 2/3])
                df_modificado['ContadorNeutro'] += np.where((df_modificado[columna] >= sextiles[0]) & (df_modificado[columna] <= sextiles[1]), 0.33, 0)
            elif num_categorias == 7:
                valor_central = np.median(valores_unicos)
                df_modificado['ContadorNeutro'] += np.where(df_modificado[columna] == valor_central, 1.0, 0)
            elif num_categorias == 8:
                cuartiles_centrales = np.quantile(valores_unicos, [0.375, 0.625])
                df_modificado['ContadorNeutro'] += np.where((df_modificado[columna] >= cuartiles_centrales[0]) & (df_modificado[columna] <= cuartiles_centrales[1]), 0.66, 0)
            elif num_categorias == 9:
                valor_central = np.median(valores_unicos)
                df_modificado['ContadorNeutro'] += np.where(df_modificado[columna] == valor_central, 1.25, 0)
            elif num_categorias == 10:
                quintiles_centrales = np.quantile(valores_unicos, [0.4, 0.6])
                df_modificado['ContadorNeutro'] += np.where((df_modificado[columna] >= quintiles_centrales[0]) & (df_modificado[columna] <= quintiles_centrales[1]), 1.0, 0)
            elif num_categorias == 11:
                sextil_central = np.quantile(valores_unicos, 0.5)
                df_modificado['ContadorNeutro'] += np.where(df_modificado[columna] == sextil_central, 1.5, 0)

    return df_modificado

# Ejemplo de uso:
df_media = contar_neutros_ponderado(df_media)

In [36]:
import pandas as pd
import numpy as np

def contar_extremos_ponderado_igual_neutro(df):
    """
    Crea una columna 'ContadorPositivo' con extremos ponderados iguales a los neutros.

    Args:
        df (pd.DataFrame): El DataFrame.

    Returns:
        pd.DataFrame: El DataFrame con la nueva columna 'ContadorPositivo'.
    """

    df_modificado = df.copy()
    df_modificado['ContadorPositivo'] = 0.0

    for columna in df_modificado.select_dtypes(include=np.number).columns:
        valores_unicos = df_modificado[columna].dropna().unique()
        num_categorias = len(valores_unicos)

        if num_categorias > 2:
            valor_minimo = np.min(valores_unicos)
            valor_maximo = np.max(valores_unicos)

            if num_categorias == 3:
                ponderacion_extremo = 0.25
            elif num_categorias == 4:
                ponderacion_extremo = 0.25
            elif num_categorias == 5:
                ponderacion_extremo = 0.5
            elif num_categorias == 6:
                ponderacion_extremo = 0.33
            elif num_categorias == 7:
                ponderacion_extremo = 1.0
            elif num_categorias == 8:
                ponderacion_extremo = 0.66
            elif num_categorias == 9:
                ponderacion_extremo = 1.25
            elif num_categorias == 10:
                ponderacion_extremo = 1.0
            elif num_categorias == 11:
                ponderacion_extremo = 1.5
            else:
                ponderacion_extremo = 0  # Valor predeterminado para otras categorias

            df_modificado['ContadorPositivo'] += np.where((df_modificado[columna] == valor_minimo) | (df_modificado[columna] == valor_maximo), ponderacion_extremo, 0)

    return df_modificado

# Ejemplo de uso:
df_media= contar_extremos_ponderado_igual_neutro(df_media)

In [37]:
#Eliminamos los valores nulos de las columnas, que hacen referencia a que el encuestado no ha querido responder. valores 77,88,99
import pandas as pd
import numpy as np

def delnulos(df):

    columnas_a_limpiar = ['pplfair','pplhlp', 'ppltrst', 'lrscale', 'stfdem','stfeco','stfgov', 'trstep', 'trstlgl', 'trstplc', 'trstplt', 'trstprl', 'trstprt','trstsci','imbgeco','imwbcnt','happy','rlgdgr']
    for columna in columnas_a_limpiar:
        if columna in df.columns:
            df[columna] = np.where(df[columna] > 70, np.nan, df[columna])
            df.dropna(subset=[columna], inplace=True)
    col_v2 = ['polintr']
    for col in col_v2:
        if col in df.columns:
            df[col] = np.where(df[col] > 4, np.nan, df[col])
            df.dropna(subset=[col], inplace=True)
    return df

delnulos(df_media)
print(df_media.isnull().sum())

cntry                   0
pplfair                 0
pplhlp                  0
ppltrst                 0
lawobey                 0
lrscale                 0
polintr                 0
stfdem                  0
stfeco                  0
stfgov                  0
trstlgl                 0
trstplc                 0
trstplt                 0
trstprl                 0
trstprt                 0
trstsci                 0
imbgeco                 0
imwbcnt                 0
happy                   0
rlgdgr                  0
cntrycat_factorizado    0
contadornegativo        0
ContadorNeutro          0
ContadorPositivo        0
dtype: int64


In [38]:
df_media.rename(columns={'lawobey': 'lw_pnd', 'trstsci': 'trtsci_pnd','cntrycat_factorizado': 'cntgrp_fc',}, inplace=True)

In [39]:
df_media.to_csv('C:/Users/Josue/4GA.DataScience/data/raw/UEmedG1.csv',index=False)

In [40]:
#%reset -f